In [ ]:
# torchrun --nproc_per_node=8 train.py\
#     --dataset coco --model maskrcnn_resnet50_fpn --epochs 26\
#     --lr-steps 16 22 --aspect-ratio-group-factor 3 --weights-backbone ResNet50_Weights.IMAGENET1K_V1

In [ ]:
import torch
import torchvision
import lightning as L
import json
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import numpy
from PIL import Image, ImageDraw
torch.set_float32_matmul_precision('medium')
import shapely

%matplotlib inline

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

device = 'cpu'
if torch.backends.mps.is_available():
    device = 'mps'
if torch.cuda.is_available():
    device = 'cuda'
print(f'Device: {device}')

In [ ]:
# weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
# model = torchvision.models.detection.maskrcnn_resnet50_fpn_v2(weights=weights)
# model = model.train().half().to(device)


In [ ]:
from torchvision.tv_tensors import BoundingBoxes
from torchvision.transforms import v2


# train_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/train/'
# val_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/val/'

# train_path = '/mnt/d/ADPKD Daten/dataset/train/'
# val_path = '/mnt/d/ADPKD Daten/dataset/val/'

train_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/train/'
val_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/val/'



class ADPKDDataset(torch.utils.data.Dataset):
    def __init__(self, path, subset):
        self.cls_id = 1
        self.image_list = glob.glob(path + 'img/' + '*.jpg')
        self.mask_list = glob.glob(path + 'mask/' + '*.png')
        self.boxes = []
        self.labels = []
        self.masks = []
        # img_files = [m.split('/')[-1].split('.')[0] for m in self.image_list]
        # mask_files = [m.split('/')[-1].split('.')[0] for m in self.mask_list]
        # print(set(img_files) - set(mask_files))
        # print(set(mask_files) - set(img_files))
        print(f'Found {len(self.image_list)} images and {len(self.mask_list)} masks')
        with open(f'dataset/{subset}.json', 'r') as file:
            self.annotations = json.load(file)
        for imgIdx, image in enumerate(self.image_list):
            # print(imgIdx, image)
            labels = []
            boxes = []
            masks = []
            image_field = list(filter(lambda x: x['file_name'] == image.split('/')[-1], self.annotations['images']))[0]
            id = image_field['id']
            img_height = image_field['height']
            img_width = image_field['width']
            annotations = list(filter(lambda x: x['image_id'] == id, self.annotations['annotations']))
            for annotationIdx, annotation in enumerate(annotations):
                # print(annotationIdx)
                xmin, ymin, width, height = annotation['bbox']
                if(annotation['area'] < 5.0):
                    # print(image)
                    continue
                # print(xmin, ymin, width, height)
                boxes.append([int(xmin), int(ymin), int(xmin + width), int(ymin + height)])
                labels.append(1)  # fixed as it only exists one class
                mask_image = Image.new('L', (img_width, img_height), 0)
                if len(annotation['segmentation']) <= 0:
                    print('Empty segementation container')
                    continue
                ImageDraw.Draw(mask_image).polygon(annotation['segmentation'][0], outline=1, fill=1)
                mask = (np.asarray(mask_image).copy() * 255).astype(np.uint8)
                mask = cv2.resize(mask, (512, 512))
                masks.append(mask)

            # print(labels, len(labels))
            # print(boxes, len(boxes))
            box_tensor = BoundingBoxes(boxes, format="XYXY", canvas_size=(img_height, img_width))
            self.boxes.append(box_tensor)
            self.labels.append(labels)
            self.masks.append(masks)
        
        self.transform = v2.Compose([
            v2.ToPILImage(),
            v2.Resize((512, 512)),
            v2.ToTensor()
        ])

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_list[idx], cv2.IMREAD_COLOR)
        img_tensor = self.transform(img)
        # mask = cv2.imread(self.mask_list[idx], cv2.IMREAD_GRAYSCALE)
        masks = self.masks[idx]
        # mask_tensor = self.transform(masks)
        #print(type(self.masks[idx]))
        mask_tensor = torch.as_tensor(self.masks[idx]) / 255.0
        # print(self.boxes[idx])
        boxes_tensor = self.transform(self.boxes[idx])
        labels_tensor = torch.Tensor(self.labels[idx]).to(torch.int64)
        return img_tensor, {'boxes': boxes_tensor, 'labels': labels_tensor, 'masks': mask_tensor}


In [ ]:
train_dataset = ADPKDDataset(train_path, 'train')
val_dataset = ADPKDDataset(val_path, 'val')

In [ ]:
print(train_dataset.masks[0][0].max())

In [ ]:
plt.imshow(train_dataset.masks[1][2], 'gray')

In [ ]:
img, targets = train_dataset[0]

In [ ]:
targets['masks'].shape

In [ ]:
np_img = np.asarray(torchvision.transforms.ToPILImage()(img)).astype(np.uint8)
for box in targets['boxes']:
    np_img = cv2.rectangle(np_img, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 0, 0), 1)

print(len(targets['masks']))
print(targets['masks'][0].shape)
print(targets['masks'][0], targets['masks'][0].max(), targets['masks'][0].min())
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 20))
ax[0].imshow(np_img)
ax[1].imshow(np.asarray(torchvision.transforms.ToPILImage()(targets['masks'][0])).astype(np.uint8), 'gray')
ax[2].imshow(np.asarray(torchvision.transforms.ToPILImage()(targets['masks'][1])).astype(np.uint8), 'gray')

In [ ]:
targets['masks'][1].max()

In [ ]:
class ADPKDDataModule(L.LightningDataModule):
    def __init__(self):
        super().__init__()
    
    def setup(self, stage):
        print(f'{stage}')
        self.train = ADPKDDataset(train_path, 'train')
        self.val = ADPKDDataset(val_path, 'val')

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=4, collate_fn=collate_fn)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=4, collate_fn=collate_fn)
    
    def on_before_batch_transfer(self, batch, dataloader_index):
        images, targets = batch
        # print(targets)
        return torch.stack(images), targets

In [ ]:
from torchvision.models.detection.backbone_utils import _resnet_fpn_extractor, _validate_trainable_layers
from torchvision.models.detection._utils import overwrite_eps
from torchvision.models.detection.mask_rcnn import MaskRCNN


class ADPKDModel(L.LightningModule):
    def __init__(self):
        super().__init__()
        weights_backbone = torchvision.models.ResNet50_Weights.IMAGENET1K_V1
        # norm_layer = torch.nn.BatchNorm2d
        trainable_backbone_layers = _validate_trainable_layers(False, 5, 5, 3)
        backbone = torchvision.models.resnet.resnet50(weights=weights_backbone, norm_layer=None, progress=True)  # if normalization is None it will be BatchNorm2D
        backbone = _resnet_fpn_extractor(backbone, trainable_backbone_layers)
        self.model = MaskRCNN(backbone, num_classes=2, rpn_score_thresh=0.80)  # background and only one class
        overwrite_eps(self.model, 0.0)
     
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        loss_dict = self.model(X, y)
        print(loss_dict)
        for k, v in loss_dict.items():
            self.log(k, float(v))
        loss = sum(loss for loss in loss_dict.values())
        self.log("train_loss", float(loss))
        return loss

    def configure_optimizers(self):
        return torch.optim.SGD(self.model.parameters(), lr=0.002, momentum=0.9, weight_decay=1e-4)  # lower lr as in reference in order to avoid NaN gradients

In [ ]:
# logger = L.pytorch.loggers.MLFlowLogger(experiment_name="lightning_logs", tracking_uri="file:./mlruns")
# ADPKDMRCNN = ADPKDModel()
# data_module = ADPKDDataModule()

In [ ]:
# trainer = L.Trainer(max_epochs=30, logger=logger)
# trainer.fit(model=ADPKDMRCNN, datamodule=data_module)

In [ ]:
# trainer.save_checkpoint('MaskRCNN_30_epochs.ckpt')

In [ ]:
model = ADPKDModel.load_from_checkpoint('MaskRCNN_30_epochs.ckpt')
model = model.eval().to(device)

In [ ]:
X, y = val_dataset[1]
X

In [ ]:
np_img = np.asarray(torchvision.transforms.ToPILImage()(X)).astype(np.uint8)
plt.imshow(np_img)

In [ ]:
y_hat = model(X.unsqueeze(0).to(device))

In [ ]:
y_hat

In [ ]:
# np_img = np.asarray(torchvision.transforms.ToPILImage()(X)).astype(np.uint8)
y_hat = y_hat[0]
for box in y_hat['boxes']:
    np_img = cv2.rectangle(np_img, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 0, 0), 1)

print(len(y_hat['masks']))
print(y_hat['masks'][0].shape)
print(y_hat['masks'][0], y_hat['masks'][0].max(), y_hat['masks'][0].min())
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 20))
ax[0].imshow(np_img)
ax[1].imshow(np.asarray(torchvision.transforms.ToPILImage()(y_hat['masks'][0])).astype(np.uint8), 'gray')
ax[2].imshow(np.asarray(torchvision.transforms.ToPILImage()(y_hat['masks'][1])).astype(np.uint8), 'gray')

In [ ]:
y_hat.keys()

In [ ]:
y_hat['boxes'][0]

In [ ]:
threshold = 0.0
offset = 5
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20, 20))

np_img = np.asarray(torchvision.transforms.ToPILImage()(X)).astype(np.uint8)
ax[0].imshow(np_img)

print(y_hat['scores'].max(), y_hat['scores'].mean(), y_hat['scores'].min())

for shapeIdx, score in enumerate(y_hat['scores']):
    if score >= threshold:
        # print(score)
        box = y_hat['boxes'][shapeIdx]
        # print(box)
        xmin, ymin, xmax, ymax = int(box[0]) - offset, int(box[1])- offset,int(box[2])+ offset, int(box[3]) + offset
        np_img = cv2.rectangle(np_img, (xmin, ymin), (xmax, ymax), (255, 0, 0), 1)
        mask = np.asarray(v2.ToPILImage()(y_hat['masks'][shapeIdx].detach().cpu()))
        mask_area = mask[ymin:ymax, xmin:xmax]
        ret2, th2 = cv2.threshold(mask_area, 0, 255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, hier = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        contours_area, hier_area = cv2.findContours(th2, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        np_img = cv2.drawContours(np_img, contours_area, 0, (0, 0, 255), 1)


ax[1].imshow(np_img)

In [ ]:
mask_test = mask[ymin-5:ymax+5, xmin-5:xmax+5]

print(mask_test.max(), mask_test.min(), mask_test.mean())
ret2, th2 = cv2.threshold(mask_test, 0, 255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)
plt.imshow(mask_test, 'gray')

In [ ]:
plt.imshow(th2, 'gray')

In [ ]:
contours_area, hier_area = cv2.findContours(th2, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
print(type(mask_test))

In [ ]:
contours_area

In [ ]:
plt.imshow(cv2.drawContours(cv2.cvtColor(mask_test.astype(np.uint8), cv2.COLOR_GRAY2RGB), contours_area, 0, (0, 0, 255), 1))

In [ ]:
plt.imshow(mask, 'gray')

In [ ]:
from torchvision.tv_tensors import BoundingBoxes
from torchvision.transforms import v2


# train_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/train/'
# val_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/val/'

# train_path = '/mnt/d/ADPKD Daten/dataset/train/'
# val_path = '/mnt/d/ADPKD Daten/dataset/val/'

train_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/train/'
val_path = '/Volumes/External/Sicherungen/basil-backup/dataset-training/val/'



class ADPKDSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, path, subset):
        self.cls_id = 1
        self.image_list = glob.glob(path + 'img/' + '*.jpg')
        self.mask_list = glob.glob(path + 'mask/' + '*.png')
        self.images = []
        self.masks = []
        print(f'Found {len(self.image_list)} images and {len(self.mask_list)} masks')
        with open(f'dataset/{subset}.json', 'r') as file:
            self.annotations = json.load(file)
        for _, image in enumerate(self.image_list):
            img = cv2.cvtColor(cv2.imread(image, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            image_field = list(filter(lambda x: x['file_name'] == image.split('/')[-1], self.annotations['images']))[0]
            id = image_field['id']
            img_height = image_field['height']
            img_width = image_field['width']
            annotations = list(filter(lambda x: x['image_id'] == id, self.annotations['annotations']))
            for _, annotation in enumerate(annotations):
                # print(annotationIdx)
                xmin, ymin, width, height = annotation['bbox']
                xmin = int(xmin)
                ymin = int(ymin)
                xmax = int(xmin + width)
                ymax = int(ymin + height)
                if(annotation['area'] < 5.0):
                    continue
                cut_img = img[ymin:ymax, xmin:xmax]
                cut_img = cv2.resize(cut_img, (384, 384))
                self.images.append(cut_img)
                mask_image = Image.new('L', (img_width, img_height), 0)
                if len(annotation['segmentation']) <= 0:
                    print('Empty segementation container')
                    continue
                ImageDraw.Draw(mask_image).polygon(annotation['segmentation'][0], outline=1, fill=1)
                mask = (np.asarray(mask_image).copy() * 255).astype(np.uint8)
                mask = mask[ymin:ymax, xmin:xmax]
                mask = cv2.resize(mask, (384, 384))
                self.masks.append(mask)
        print(f'Cutted {len(self.images)} images and {len(self.masks)} masks')
        
        self.transform = v2.Compose([
            v2.ToPILImage(),
            # v2.Resize((512, 512)),
            v2.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # img = cv2.imread(self.images[idx], cv2.IMREAD_COLOR)
        img_tensor = self.transform(self.images[idx])
        mask_tensor = torch.as_tensor(self.masks[idx]) / 255.0
        return img_tensor, mask_tensor


In [ ]:
train_dataset = ADPKDSegmentationDataset(train_path, 'train')
val_dataset = ADPKDSegmentationDataset(val_path, 'val')

In [ ]:
len(train_dataset)

In [ ]:
img, targets = train_dataset[6]

In [ ]:
plt.imshow(np.asarray(torchvision.transforms.ToPILImage()(img)).astype(np.uint8))

In [ ]:
class ADPKDSegmentationDataModule(L.LightningDataModule):
    def __init__(self):
        super().__init__()
    
    def setup(self, stage):
        print(f'{stage}')
        self.train = ADPKDSegmentationDataset(train_path, 'train')
        self.val = ADPKDSegmentationDataset(val_path, 'val')

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=4, collate_fn=collate_fn)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=4, collate_fn=collate_fn)
    
    def on_before_batch_transfer(self, batch, dataloader_index):
        images, targets = batch
        # print(targets)
        return torch.stack(images), torch.stack(targets)

In [ ]:
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss
from segmentation_models_pytorch.utils.metrics import IoU

class ADPKDSegmentationModel(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(encoder_name='timm-resnest26d', encoder_weights='imagenet', in_channels=3, classes=1, activation='sigmoid')
        self.loss = DiceLoss(mode='binary')
        self.metrics = [IoU(threshold=0.90)]
     
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        pred = self.model(X)
        loss = self.loss(pred, y)
        self.log("train_loss", float(loss))
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=0.0001)

In [ ]:
# logger = L.pytorch.loggers.MLFlowLogger(experiment_name="lightning_logs", tracking_uri="file:mlruns")
# ADPKDUnet = ADPKDSegmentationModel.load_from_checkpoint('ADPKDUnet_10_epochs.ckpt')
# data_module = ADPKDSegmentationDataModule()

In [ ]:
# trainer = L.Trainer(max_epochs=10, logger=logger)
# trainer.fit(model=ADPKDUnet, datamodule=data_module)

In [ ]:
# trainer.save_checkpoint('ADPKDUnet_10_epochs_2.ckpt')

In [ ]:
seg_model = ADPKDSegmentationModel.load_from_checkpoint('ADPKDUnet_10_epochs_2.ckpt')
seg_model.half().eval().to(device)

In [ ]:
y_hat = seg_model(img.unsqueeze(0).half().to(device)).squeeze()

In [ ]:
plt.imshow(np.asarray(torchvision.transforms.ToPILImage()(y_hat)).astype(np.uint8), 'gray')

In [ ]:
mask = np.asarray(v2.ToPILImage()(y_hat.detach().cpu())).astype(np.uint8)
_, mask_th = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
try:
    contours_area, _ = cv2.findContours(mask_th, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
except IndexError:
    print('Index error')

In [ ]:
plt.imshow(cv2.drawContours(cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB), contours_area, 0, (255, 0, 0), 2))